# SatQuery AI: Multimodal Remote Sensing Adaptation on BigEarthNet.txt
**Problem Statement 26167 | ISRO / Department of Space | SIH 2026**

This notebook demonstrates domain adaptation of vision-language backbones using paired Sentinel-1 (C-band SAR) and Sentinel-2 (Multispectral Optical) imagery with multi-label land cover annotations from `BigEarthNet.txt`.

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as T
import numpy as np
from PIL import Image

print(f"CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Multimodal Optical-SAR Dual-Stream Architecture
We fuse the optical RGB/NIR bands with SAR $VV$ and $VH$ backscatter polarization channels.

In [ ]:
class MultimodalRemoteSensingBackbone(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        # Optical stream (Sentinel-2 VNIR)
        self.optical_encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # SAR stream (Sentinel-1 VV/VH backscatter)
        self.sar_encoder = nn.Sequential(
            nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Cross-modal projection & fusion head
        self.fusion_head = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, 19)  # 19 CORINE BigEarthNet classes

    def forward(self, x_opt, x_sar):
        f_opt = self.optical_encoder(x_opt)
        f_sar = self.sar_encoder(x_sar)
        fused = torch.cat([f_opt, f_sar], dim=-1)
        emb = torch.relu(self.fusion_head(fused))
        logits = self.classifier(emb)
        return logits, emb

model = MultimodalRemoteSensingBackbone().to(device)
print("Multimodal Optical-SAR Backbone initialized.")

## 2. Remote-Sensing Contrastive Loss (InfoNCE + Multi-Label BCE)
Combines symmetric contrastive text-image alignment with land-cover classification.

In [ ]:
criterion_cls = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# Verification on dummy batch
x_opt = torch.randn(4, 3, 120, 120).to(device)
x_sar = torch.randn(4, 2, 120, 120).to(device)
labels = torch.randint(0, 2, (4, 19)).float().to(device)

logits, emb = model(x_opt, x_sar)
loss = criterion_cls(logits, labels)
print(f"Dummy forward pass loss: {loss.item():.4f}")
print(f"Embedding shape: {emb.shape}")